In [4]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"

In [5]:
locations = pd.read_csv("../data/locations.csv")
locations

,id,lat,lon
0,1,30.222,31.732
1,2,30.238,31.700
2,3,29.853,31.387
3,4,29.985,30.963
4,5,29.941,30.908
...,...,...,...
360,361,25.634,32.716
361,362,27.358,31.799
362,363,26.727,32.193
363,364,25.966,33.191


In [11]:
def fetch_all_air_qualities(locations_df):
    params = {
        # Pass full lists instead of single floats
        "latitude": locations_df["lat"].tolist(),
        "longitude": locations_df["lon"].tolist(),
        "hourly": [
            "pm10",
            "pm2_5",
            "us_aqi",
            "us_aqi_pm2_5",
            "us_aqi_pm10",
            "us_aqi_nitrogen_dioxide",
        ],
        "past_days": 92,
        "domains": "cams_global",
        "timeformat": "unixtime",
    }

    # Single network request for ALL locations
    responses = openmeteo.weather_api(url, params=params)

    data_list = []

    # Iterate over the API responses and location IDs in sync
    for dist_id, response in zip(locations_df["id"], responses):
        hourly = response.Hourly()

        data = {
            "id": dist_id,
            "pm10": hourly.Variables(0).ValuesAsNumpy().mean(),
            "pm2_5": hourly.Variables(1).ValuesAsNumpy().mean(),
            "us_aqi": hourly.Variables(2).ValuesAsNumpy().mean(),
            "us_pm2_5": hourly.Variables(3).ValuesAsNumpy().mean(),
            "us_pm10": hourly.Variables(4).ValuesAsNumpy().mean(),
            "us_no2": hourly.Variables(5).ValuesAsNumpy().mean(),
        }
        data_list.append(data)

    return pd.DataFrame(data_list)

In [13]:
# Chunking example if you have many locations
import time

CHUNK_SIZE = 100
df_list = []

for i in range(0, len(locations), CHUNK_SIZE):
    chunk = locations.iloc[i : i + CHUNK_SIZE]
    df_list.append(fetch_all_air_qualities(chunk))
    time.sleep(60)

aq_df = pd.concat(df_list, ignore_index=True)

In [14]:
aq_df

,id,pm10,pm2_5,us_aqi,us_pm2_5,us_pm10,us_no2
0,1,24.705069,15.832259,74.846085,62.892975,22.433071,4.978726
1,2,24.705069,15.832259,74.846085,62.892975,22.433071,4.978726
2,3,36.463833,19.930069,79.891678,70.672394,31.697279,9.088648
3,4,54.698410,20.782001,78.130989,72.362312,44.380844,3.544936
4,5,54.698410,20.782001,78.130989,72.362312,44.380844,3.544936
...,...,...,...,...,...,...,...
360,361,247.036728,56.871864,152.158752,141.079605,148.579758,1.692932
361,362,363.516937,67.627541,260.166168,148.501328,259.406311,1.615775
362,363,429.727020,76.577400,321.070587,159.843399,321.020508,1.140080
363,364,131.597290,32.567398,98.238861,96.695648,88.139984,0.631996


In [15]:
aq_df.to_csv("../data/airquality.csv", sep=",", index=False, header=True)